# Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import layers,models
from tensorflow.keras.applications import MobileNetV2

# Load dataset

In [2]:
dataset_path = "dataset"

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset ="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224, 224),
    batch_size=32
)

Found 6652 files belonging to 6 classes.
Using 5322 files for training.
Found 6652 files belonging to 6 classes.
Using 1330 files for validation.


In [3]:
class_names = train_ds.class_names
print(class_names)

['Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_healthy']


# Normalize Images

In [4]:
normalization_layer = layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x,y:(normalization_layer(x),y))
val_ds = val_ds.map(lambda x,y:(normalization_layer(x),y))

# Load Pretrained Model

We use MobileNetV2 as the feature extractor.

In [6]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
                         )
base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


# Build the Model

In [7]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(class_names),activation='softmax')
])

# Compile Model

In [9]:
model.compile(
    optimizer ='adam',
    loss ='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [10]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

Epoch 1/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 57s 323ms/step - accuracy: 0.8482 - loss: 0.4218 - val_accuracy: 0.9038 - val_loss: 0.2381
Epoch 2/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 51s 303ms/step - accuracy: 0.9329 - loss: 0.1867 - val_accuracy: 0.9135 - val_loss: 0.2323
Epoch 3/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 53s 318ms/step - accuracy: 0.9487 - loss: 0.1437 - val_accuracy: 0.9421 - val_loss: 0.1568
Epoch 4/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 51s 305ms/step - accuracy: 0.9632 - loss: 0.1135 - val_accuracy: 0.9526 - val_loss: 0.1402
Epoch 5/5
167/167 ━━━━━━━━━━━━━━━━━━━━ 50s 299ms/step - accuracy: 0.9720 - loss: 0.0838 - val_accuracy: 0.9549 - val_loss: 0.1343


In [11]:
model.save("model/plant_model.h5")